# Probeprüfung AP01: Management & Nutzung relationaler Daten (Probeprüfung)

Aufgebaut wie die Prüfung *AP01* — gleiche Teile, gleiche Punkte, gleiches Bewertungsschema —, aber mit der
fiktiven Datenbank der Fitnesskette **MoveFit** statt `cinema.db`.

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

| Teil | Thema | Aufgaben | Punkte |
|---|---|---|---|
| A | Konzeptfragen: Datenmodellierung & Design | A1–A3 | 12 |
| B | SQL-Fehleranalyse (fitness.db) | B1–B2 | 16 |
| C | SQL-Abfragen schreiben (fitness.db) | C1–C3 | 12 |
| **Total** | | **8 Aufgaben** | **40** |

### Bewertungsschema

| Kategorie | Beschreibung | Bewertung | 4 Pkt | 8 Pkt |
|---|---|---|---|---|
| Korrekt | Vollständig und fachlich einwandfrei; SQL liefert das richtige Ergebnis | 100% | 4 | 8 |
| Minimale Mängel | Kleine Fehler (z. B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau | 75% | 3 | 6 |
| Mittlere Mängel | Grössere Fehler (z. B. fehlende Spalten, falsche Filterung) oder Begründung fehlt | 50% | 2 | 4 |
| Gravierende Mängel | Ansatz im Kern falsch (z. B. falscher JOIN-Typ, fehlende Tabelle) | 25% | 1 | 2 |
| Nicht lauffähig | Antwort fehlt oder Code nicht lauffähig | 0% | 0 | 0 |

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege gelten als korrekt.

*Empfohlene Bearbeitungszeit: 60 Minuten. Arbeite «open book» (Factsheet, Referenz, eigene Übungen), aber ohne KI —
wie in der Prüfung.*

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup aufgebaut und getestet.

**Backup:** `Data/fitness.db` (SQLite) — wird automatisch verwendet, falls PostgreSQL nicht erreichbar ist.

**Tabellen:** `studios`, `courses`, `trainers`, `sessions`, `members`, `bookings` — Beschreibung und Datenmodell in
`Data/fitness_beschreibung.md`.

**Antworten auf Konzeptfragen** tragen Sie direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [ ]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

In [ ]:
# Zugangsdaten (Werte aus docker-compose.yml, über Umgebungsvariablen überschreibbar)
user     = os.getenv('PG_USER', 'pgadmin')
password = os.getenv('PG_PASSWORT', 'geheim')
port     = os.getenv('PG_PORT', '5432')
database = os.getenv('PG_DATABASE', 'postgres')

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in (os.getenv('PG_HOST'), 'db', 'localhost'):
    if not candidate:
        continue
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

if host:
    db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'
else:
    # Backup: SQLite-Datei verwenden (gleiche Tabellen und Inhalte)
    print('Keine PostgreSQL-Verbindung gefunden -> verwende das SQLite-Backup Data/fitness.db')
    db_connection_url = 'sqlite:///Data/fitness.db'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Verbindung:', engine.dialect.name, '| Host:', host)
engine.dispose()

In [ ]:
# Tabellen aus der SQLite-Datei fitness.db nach PostgreSQL übertragen
tables = ['studios', 'courses', 'trainers', 'sessions', 'members', 'bookings']

if db_connection_url.startswith('postgresql'):
    sqlite_engine = create_engine('sqlite:///Data/fitness.db')
    engine = create_engine(db_connection_url)
    for table in tables:
        df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
        df.to_sql(table, engine, if_exists='replace', index=False)
        print(f'{table:<10}{len(df):>6} Zeilen')
    sqlite_engine.dispose()
    engine.dispose()

---
**▶ Ab hier beginnt die bewertete Probeprüfung — 40 Punkte**

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `fitness.db` werden nicht gewertet.*

### A1 (4 Punkte)

Ein Kollege schlägt vor, die Spalten `studio_name` und `studio_city` direkt in die Tabelle `sessions` aufzunehmen:
*«Dann sparen wir uns bei jeder Auswertung den JOIN mit `studios`.»*

Ist das eine gute Idee? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Betrieb der Fitnesskette.

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Das Datenteam analysiert die Spalte `members.email` und findet unter anderem folgende Werte:

```
anna.mueller@example.ch, NULL (15-mal), '' (leerer Text), 'n/a', 'jonas.huber@example'
```

Beschreiben Sie die konkreten Datenprobleme und nennen Sie die passenden Data-Quality-Eigenschaften aus den
Vorlesungsfolien (z. B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Der Kurstermin `SE0005` hat eine Kapazität von 12 Plätzen, in `bookings` gibt es dafür aber 13 nicht stornierte
Buchungen. Eine Kollegin sagt: *«Das ist ein Datenfehler, die 13. Buchung muss gelöscht werden.»*
Ist das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis.*

### B1 (8 Punkte)

**Ziel:** Anzahl Buchungen pro Kurs (`title`) — für **alle** Kurse in der Datenbank.

```sql
SELECT c.title, COUNT(*) AS booking_count
FROM sessions s
JOIN bookings b ON s.sessionid = b.sessionid
JOIN courses c  ON s.courseid  = c.courseid
GROUP BY c.title;
```

Die Tabelle `courses` enthält **15 Kurse**. Tatsächliches Ergebnis der SQL-Abfrage: nur **14 Zeilen**.

Die Leiterin Kursplanung fragt: *«Zeig mir auch Kurse, die noch nie gebucht wurden.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 15 Kurse** erscheinen und ein Kurs ohne Buchungen
korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen
""", con=engine)

engine.dispose()
df_B1

### B2 (8 Punkte)

**Ziel:** Tatsächlich erzielter Umsatz pro Studio (`name`). Stornierte Buchungen (`status = 'cancelled'`) werden
vollständig zurückerstattet; Buchungen mit `no-show` bleiben bezahlt.

```sql
SELECT st.name, SUM(s.base_price) AS revenue
FROM studios st
JOIN sessions s ON st.studioid  = s.studioid
JOIN bookings b ON s.sessionid  = b.sessionid
GROUP BY st.name
ORDER BY revenue DESC;
```

Tatsächliches Ergebnis (Ausschnitt):

```
               name   revenue
0  MoveFit Oerlikon   14523.0
1   MoveFit Bahnhof   10548.0
...
```

Die Finanzabteilung meldet, dass die Bankeingänge deutlich tiefer liegen.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage (Umsatz auf 2 Dezimalstellen gerundet). Begründen Sie jede
Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen
""", con=engine)

engine.dispose()
df_B2

---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `fitness.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.

### C1 (4 Punkte)

Schreiben Sie eine SQL-Abfrage, die alle Mitglieder (`memberid`, `firstname`, `lastname`, `email`) ausgibt, deren
E-Mail **fehlt oder ungültig** ist. Als ungültig gilt eine E-Mail, die leer ist, kein `@` enthält oder nach dem `@`
keinen Punkt hat. Sortieren Sie nach `memberid`. Begründen Sie Ihre Filterbedingung — insbesondere den Umgang mit NULL.

*Hinweis:* Enthält eine Abfrage `%`, übergeben Sie sie an PostgreSQL als `text("""...""")` (siehe Übung 10).

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen
""", con=engine)

engine.dispose()
df_C1

### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Preis (`paid_price`) pro Kurskategorie (`category`)? Geben Sie `category`
und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`.
Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen
""", con=engine)

engine.dispose()
df_C2

### C3 (4 Punkte)

Welche Trainer haben **mehr als 15** Kurstermine geleitet? Geben Sie den vollen Namen als `trainer_name`
(Vorname Leerschlag Nachname) und die Anzahl Termine als `session_count` aus, absteigend nach `session_count` und bei
Gleichstand nach `trainer_name`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen
""", con=engine)

engine.dispose()
df_C3

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')